# SoccerNet Ball Detection + Tracking 1×4 Benchmark — YOLO only

This notebook is split from the previous 2×4 benchmark to avoid long combined training runs.

It runs exactly one detector with four trackers:

| Detector | Tracker |
|---|---|
| YOLO | Greedy online tracker from `ball-detection.ipynb` |
| YOLO | Viterbi / dynamic-programming tracker from `ball_tracking_viterbi_notebook.ipynb` |
| YOLO | SORT tracker |
| YOLO | OC-SORT-style observation-centric SORT tracker |

Main outputs:

- candidate detections for YOLO,
- final tracks for YOLO × 4 trackers,
- MOT-format prediction files,
- a comparison table with standard MOT metrics, HOTA-style metrics, and inference-time benchmark columns.

Default matrix: **1 detector × 4 trackers = 4 runs**.

Timing benchmark:

- `detection_time_sec`: detector candidate generation time per sequence.
- `tracker_time_sec`: tracker runtime per sequence.
- `total_inference_time_sec`: detection + tracking time for one detector × tracker pair.
- `detect_fps`, `track_fps`, `end_to_end_fps`: frame throughput.

For accurate detector timing, keep `REUSE_CANDIDATES_FOR_BENCHMARK = False` in the configuration cell. If it is `True`, detection time measures CSV cache loading rather than model inference.

Training is separated: this notebook does **not** train or load DETR.


In [ ]:
# Kaggle / Colab setup for YOLO-only benchmark
# If your environment already has these packages, this cell is safe to rerun.
!pip -q install ultralytics opencv-python-headless pyyaml tqdm pandas scipy pillow


In [ ]:
from pathlib import Path
import os
import re
import math
import json
import shutil
import random
import subprocess
import time
import configparser
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import cv2
import yaml
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from scipy.optimize import linear_sum_assignment

from IPython.display import display, Video, HTML

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 1. Configuration

Edit the next code cell before running the notebook. That cell controls:

- dataset/checkpoint paths,
- the single enabled detector and four trackers,
- YOLO training settings,
- candidate inference thresholds,
- timing benchmark behavior,
- tracker hyperparameters via the tracker parameter dataclasses later in the notebook.

This split notebook intentionally runs only `yolo`. To compare both models, run the `yolo` notebook and the other 1×4 notebook separately, then compare their CSV summaries.

For quick debugging, set `MAX_EVAL_SEQS = 1`, use low epochs, or temporarily reduce `RUN_TRACKERS`.


In [ ]:
# =========================
# Data and output paths
# =========================
def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    marker_sets = [
        ("data", "outputs", "notebooks"),
        ("data", "notebooks"),
        ("README.md", "notebooks"),
    ]
    for candidate in [current, *current.parents]:
        for markers in marker_sets:
            if all((candidate / marker).exists() for marker in markers):
                return candidate
    return current


def has_soccernet_sequences(path: Path) -> bool:
    path = Path(path)
    return path.exists() and any(path.rglob("seqinfo.ini"))


def resolve_kaggle_tracking_train_dir() -> Path:
    input_root = Path("/kaggle/input")
    candidates = [
        input_root / "notebooks" / "vtnhan2906" / "soccernet-tracking" / "train",
        input_root / "soccernet-tracking" / "train",
        input_root / "soccernet" / "tracking" / "train",
    ]
    for candidate in candidates:
        if has_soccernet_sequences(candidate):
            return candidate

    if input_root.exists():
        for seqinfo in input_root.rglob("seqinfo.ini"):
            sequence_dir = seqinfo.parent
            if sequence_dir.parent.name.lower() == "train":
                return sequence_dir.parent
    raise FileNotFoundError(
        "Could not find SoccerNet tracking train split under /kaggle/input. "
        "Expected a folder like soccernet-tracking/train containing SNMOT sequence folders."
    )


def resolve_local_tracking_train_dir(project_root: Path) -> Path:
    candidates = [
        project_root / "data" / "raw" / "tracking" / "train",
        project_root / "data" / "tracking" / "train",
    ]
    for candidate in candidates:
        if has_soccernet_sequences(candidate):
            return candidate
    raise FileNotFoundError(
        "Could not find local SoccerNet tracking train split. "
        "Expected data/raw/tracking/train or data/tracking/train containing SNMOT sequence folders."
    )


PROJECT_ROOT = find_project_root()
IS_KAGGLE = Path("/kaggle/working").exists()
DATA_ROOT = resolve_kaggle_tracking_train_dir() if IS_KAGGLE else resolve_local_tracking_train_dir(PROJECT_ROOT)
WORK_ROOT = (
    Path("/kaggle/working") / "ball_yolo_1x4_tracking_benchmark"
    if IS_KAGGLE
    else PROJECT_ROOT / "outputs" / "ball_tracking" / "yolo_1x4_tracking_benchmark"
)

# Optional: set these to an existing checkpoint to skip training.
YOLO_EXISTING_CKPT = None
DETR_EXISTING_DIR = None

# Experiment matrix. This split notebook intentionally runs only YOLO.
EXPERIMENT_TAG = "yolo_1x4"
RUN_DETECTORS = ["yolo"]
RUN_TRACKERS = ["greedy", "viterbi", "sort", "ocsort"]

# Train/val split by sequence.
VAL_RATIO = 0.10
MAX_TRAIN_SEQS = None       # None = all train split sequences
MAX_EVAL_SEQS = None        # None = all val split sequences. Use 1 or 2 for debug.
SELECT_EVAL_SEQ_NAMES = None  # Example: ["SNMOT-168"]. Overrides MAX_EVAL_SEQS if not None.

# Detection dataset options.
INCLUDE_NEGATIVE_FRAMES = True
MIN_BOX_SIZE = 1.0
SYMLINK_IMAGES = True
ALLOW_COPY_IMAGE_FALLBACK = False
REBUILD_YOLO_DATASET = False

# YOLO training/inference.
YOLO_WEIGHTS = "yolov8n.pt"  # Try yolo11n.pt/yolo11s.pt for faster debug; yolo11l.pt for stronger result.
YOLO_IMGSZ = 1280
YOLO_EPOCHS = 15
YOLO_BATCH = 32
YOLO_DEVICE_TRAIN = "0"      # "0,1" if you want multi-GPU training on Kaggle T4x2.
YOLO_DEVICE_INFER = 0
YOLO_WORKERS = 2

# DETR training/inference. Not used in this YOLO-only notebook unless you manually add "detr" to RUN_DETECTORS.
DETR_BASE_MODEL = "facebook/detr-resnet-50"
DETR_EPOCHS = 3
DETR_BATCH = 2
DETR_LR = 1e-5
DETR_BACKBONE_LR = 1e-6
DETR_WEIGHT_DECAY = 1e-4
DETR_NUM_WORKERS = 2

# Shared inference settings.
INFER_CONF = 0.005
INFER_IOU = 0.60
TOP_K = 15
INFER_BATCH = 16

# Timing benchmark.
# False = rerun detector inference and measure real detector time.
# True  = reuse existing candidate CSVs if available; faster, but detection time then means cache-read time.
REUSE_CANDIDATES_FOR_BENCHMARK = False
TIMING_SYNC_CUDA = True

# Evaluation.
MOT_IOU_THRESHOLD = 0.50
HOTA_THRESHOLDS = np.arange(0.05, 0.96, 0.05)
CENTER_HIT_PX = 20

# Output paths.
YOLO_DATA_ROOT = WORK_ROOT / "yolo_dataset"
RUNS_ROOT = WORK_ROOT / "runs"
DETR_RUN_DIR = WORK_ROOT / "detr_ball"
CAND_ROOT = WORK_ROOT / "candidates"
TRACK_ROOT = WORK_ROOT / "tracks"
MOT_ROOT = WORK_ROOT / "mot_predictions"
VIS_ROOT = WORK_ROOT / "vis"
RESULTS_ROOT = WORK_ROOT / "results"

for p in [WORK_ROOT, YOLO_DATA_ROOT, RUNS_ROOT, DETR_RUN_DIR, CAND_ROOT, TRACK_ROOT, MOT_ROOT, VIS_ROOT, RESULTS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IS_KAGGLE:", IS_KAGGLE)
print("DATA_ROOT:", DATA_ROOT, "exists=", DATA_ROOT.exists())
print("WORK_ROOT:", WORK_ROOT)


## 1.1 Where to edit configs

Most configs are in **Section 1 / Configuration**, the code cell directly above this text. Practical map:

| What you want to change | Variable / location |
|---|---|
| Dataset path | `DATA_ROOT` |
| Output folder | `WORK_ROOT` |
| Experiment name used in CSV names | `EXPERIMENT_TAG` |
| Enabled detector | `RUN_DETECTORS = ["yolo"]` |
| Enabled trackers | `RUN_TRACKERS = ["greedy", "viterbi", "sort", "ocsort"]` |
| Debug on fewer sequences | `MAX_EVAL_SEQS`, `SELECT_EVAL_SEQ_NAMES`, `MAX_TRAIN_SEQS` |
| Skip YOLO training and use a saved weight | `YOLO_EXISTING_CKPT` |
| Skip DETR training and use a saved checkpoint dir | `DETR_EXISTING_DIR` |
| YOLO model size / training | `YOLO_WEIGHTS`, `YOLO_IMGSZ`, `YOLO_EPOCHS`, `YOLO_BATCH`, `YOLO_DEVICE_TRAIN` |
| DETR training | `DETR_BASE_MODEL`, `DETR_EPOCHS`, `DETR_BATCH`, `DETR_LR` |
| Detector inference threshold | `INFER_CONF`, `INFER_IOU`, `TOP_K`, `INFER_BATCH` |
| Accurate timing vs cached fast rerun | `REUSE_CANDIDATES_FOR_BENCHMARK` |
| Evaluation IoU / HOTA thresholds | `MOT_IOU_THRESHOLD`, `HOTA_THRESHOLDS`, `CENTER_HIT_PX` |
| Greedy tracker hyperparameters | `GreedyTrackParams` in Section 6 |
| Viterbi tracker hyperparameters | `ViterbiTrackParams` in Section 7 |
| SORT tracker hyperparameters | `SortTrackParams` in Section 8 |
| OC-SORT-style tracker hyperparameters | `OCSortTrackParams` in Section 8 |
| Render qualitative video | `SAMPLE_DETECTOR`, `SAMPLE_TRACKER`, `SAMPLE_SEQ_NAME` in Section 12 |

Important for timing: to benchmark actual detector inference, keep `REUSE_CANDIDATES_FOR_BENCHMARK = False`. If you set it to `True`, the notebook reuses `candidates/*.csv`, so the detection timing is no longer real model inference time.

This file is **YOLO-only by default**. To run the other detector, use the other split notebook instead of editing this one.


## 2. SoccerNet helpers

Important: `gt/gt.txt` is MOT-style. The object class must be recovered from `gameinfo.ini`, not from the unused columns in `gt.txt`.

In [ ]:
GT_COLUMNS = [
    "frame_id", "track_id", "x", "y", "w", "h",
    "conf", "unused1", "unused2", "unused3"
]

def parse_seqinfo(seq_dir: Path) -> Dict[str, Any]:
    ini_path = seq_dir / "seqinfo.ini"
    if not ini_path.exists():
        raise FileNotFoundError(f"Missing seqinfo.ini in {seq_dir}")

    cp = configparser.ConfigParser()
    cp.read(ini_path)
    sec = cp["Sequence"]

    return {
        "name": sec.get("name", seq_dir.name),
        "im_dir": sec.get("imDir", "img1"),
        "frame_rate": sec.getint("frameRate", fallback=25),
        "seq_length": sec.getint("seqLength"),
        "im_width": sec.getint("imWidth"),
        "im_height": sec.getint("imHeight"),
        "im_ext": sec.get("imExt", ".jpg"),
    }

def frame_path(seq_dir: Path, frame_id: int) -> Path:
    info = parse_seqinfo(seq_dir)
    return seq_dir / info["im_dir"] / f"{frame_id:06d}{info['im_ext']}"

def discover_sequences(root: Path) -> List[Path]:
    root = Path(root)
    seqs = []
    for seqinfo in root.rglob("seqinfo.ini"):
        seq_dir = seqinfo.parent
        try:
            info = parse_seqinfo(seq_dir)
            img_dir = seq_dir / info["im_dir"]
            if img_dir.exists():
                seqs.append(seq_dir)
        except Exception:
            pass
    return sorted(set(seqs), key=lambda p: p.name)

def parse_gameinfo(seq_dir: Path) -> Dict[int, str]:
    gameinfo = seq_dir / "gameinfo.ini"
    if not gameinfo.exists():
        return {}

    mapping = {}
    text = gameinfo.read_text(errors="replace")
    for line in text.splitlines():
        line = line.strip()
        m = re.match(r"trackletID_(\d+)\s*=\s*(.+)", line)
        if m:
            track_id = int(m.group(1))
            label = m.group(2).strip()
            mapping[track_id] = label
    return mapping

def get_ball_track_ids(seq_dir: Path) -> List[int]:
    mapping = parse_gameinfo(seq_dir)
    ball_ids = []
    for tid, label in mapping.items():
        if label.lower().split(";")[0].strip() == "ball":
            ball_ids.append(tid)
    return sorted(ball_ids)

def read_gt(seq_dir: Path) -> pd.DataFrame:
    gt_path = seq_dir / "gt" / "gt.txt"
    if not gt_path.exists():
        return pd.DataFrame(columns=GT_COLUMNS)
    return pd.read_csv(gt_path, header=None, names=GT_COLUMNS)

def clip_box_xywh(x, y, w, h, W, H):
    x1 = max(0.0, float(x))
    y1 = max(0.0, float(y))
    x2 = min(float(W), float(x) + float(w))
    y2 = min(float(H), float(y) + float(h))
    cw = max(0.0, x2 - x1)
    ch = max(0.0, y2 - y1)
    return x1, y1, cw, ch

def xywh_to_xyxy_arr(boxes):
    boxes = np.asarray(boxes, dtype=np.float64)
    if boxes.size == 0:
        return boxes.reshape(0, 4)
    out = boxes.copy()
    out[:, 2] = out[:, 0] + out[:, 2]
    out[:, 3] = out[:, 1] + out[:, 3]
    return out

def iou_matrix_xywh(a_xywh, b_xywh):
    a = xywh_to_xyxy_arr(a_xywh)
    b = xywh_to_xyxy_arr(b_xywh)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float64)

    ax1, ay1, ax2, ay2 = a[:, 0][:, None], a[:, 1][:, None], a[:, 2][:, None], a[:, 3][:, None]
    bx1, by1, bx2, by2 = b[:, 0][None, :], b[:, 1][None, :], b[:, 2][None, :], b[:, 3][None, :]

    ix1 = np.maximum(ax1, bx1)
    iy1 = np.maximum(ay1, by1)
    ix2 = np.minimum(ax2, bx2)
    iy2 = np.minimum(ay2, by2)

    iw = np.maximum(0.0, ix2 - ix1)
    ih = np.maximum(0.0, iy2 - iy1)
    inter = iw * ih

    area_a = np.maximum(0.0, (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]))[:, None]
    area_b = np.maximum(0.0, (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1]))[None, :]
    union = np.maximum(area_a + area_b - inter, 1e-12)
    return inter / union

def load_ball_gt_for_seq(seq_dir: Path) -> pd.DataFrame:
    info = parse_seqinfo(seq_dir)
    W, H = info["im_width"], info["im_height"]
    ball_ids = get_ball_track_ids(seq_dir)
    gt = read_gt(seq_dir)

    if not ball_ids or len(gt) == 0:
        return pd.DataFrame(columns=["frame_id", "track_id", "x", "y", "w", "h", "cx", "cy"])

    ball_gt = gt[gt["track_id"].isin(ball_ids)].copy()
    rows = []
    for _, r in ball_gt.iterrows():
        x, y, w, h = clip_box_xywh(r.x, r.y, r.w, r.h, W, H)
        if w < MIN_BOX_SIZE or h < MIN_BOX_SIZE:
            continue
        rows.append({
            "frame_id": int(r.frame_id),
            "track_id": int(r.track_id),
            "x": x,
            "y": y,
            "w": w,
            "h": h,
            "cx": x + w / 2.0,
            "cy": y + h / 2.0,
        })
    return pd.DataFrame(rows)

def ball_boxes_by_frame(seq_dir: Path) -> Dict[int, List[List[float]]]:
    gt = load_ball_gt_for_seq(seq_dir)
    out = {}
    if len(gt) == 0:
        return out
    for fid, g in gt.groupby("frame_id"):
        out[int(fid)] = g[["x", "y", "w", "h"]].astype(float).values.tolist()
    return out

def safe_link_or_copy(src: Path, dst: Path, symlink=True):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        return

    link_errors = []
    if symlink:
        try:
            os.link(src, dst)
            return
        except Exception as exc:
            link_errors.append(f"hardlink failed: {exc}")
        try:
            os.symlink(src, dst)
            return
        except Exception as exc:
            link_errors.append(f"symlink failed: {exc}")

    if ALLOW_COPY_IMAGE_FALLBACK:
        shutil.copy2(src, dst)
        return

    raise RuntimeError(
        "Could not link image without copying. Enable Windows Developer Mode/admin, "
        "or set ALLOW_COPY_IMAGE_FALLBACK=True if you accept copying images into outputs. "
        f"Source={src} Destination={dst}. " + " | ".join(link_errors)
    )

def build_frame_index(seqs: List[Path], include_negative=True) -> pd.DataFrame:
    rows = []
    for seq_dir in tqdm(seqs, desc="build frame index"):
        info = parse_seqinfo(seq_dir)
        gt_by_frame = ball_boxes_by_frame(seq_dir)
        for fid in range(1, info["seq_length"] + 1):
            img = frame_path(seq_dir, fid)
            if not img.exists():
                continue
            boxes = gt_by_frame.get(fid, [])
            if len(boxes) == 0 and not include_negative:
                continue
            rows.append({
                "seq": seq_dir.name,
                "seq_path": str(seq_dir),
                "frame_id": int(fid),
                "image_path": str(img),
                "width": int(info["im_width"]),
                "height": int(info["im_height"]),
                "boxes": boxes,
            })
    return pd.DataFrame(rows)

In [ ]:
seqs = discover_sequences(DATA_ROOT)
assert len(seqs) > 0, f"No SoccerNet sequences found under {DATA_ROOT}"

random.Random(SEED).shuffle(seqs)
n_val = max(1, int(round(len(seqs) * VAL_RATIO)))
val_seqs = sorted(seqs[:n_val], key=lambda p: p.name)
train_seqs = sorted(seqs[n_val:], key=lambda p: p.name)

if MAX_TRAIN_SEQS is not None:
    train_seqs = train_seqs[:MAX_TRAIN_SEQS]

if SELECT_EVAL_SEQ_NAMES is not None:
    name_set = set(SELECT_EVAL_SEQ_NAMES)
    eval_seqs = [s for s in val_seqs if s.name in name_set]
else:
    eval_seqs = val_seqs if MAX_EVAL_SEQS is None else val_seqs[:MAX_EVAL_SEQS]

print(f"Total sequences: {len(seqs)}")
print(f"Train sequences: {len(train_seqs)}")
print(f"Validation sequences: {len(val_seqs)}")
print(f"Evaluation sequences: {len(eval_seqs)}")
print("Eval:", [s.name for s in eval_seqs[:20]])

train_index = build_frame_index(train_seqs, include_negative=INCLUDE_NEGATIVE_FRAMES)
val_index = build_frame_index(val_seqs, include_negative=INCLUDE_NEGATIVE_FRAMES)
eval_index = build_frame_index(eval_seqs, include_negative=INCLUDE_NEGATIVE_FRAMES)

print("train frames:", len(train_index), "val frames:", len(val_index), "eval frames:", len(eval_index))
display(train_index.head())

## 3. Build YOLO dataset and train/load YOLO detector

This section is skipped automatically when `RUN_DETECTORS` does not contain `"yolo"`.


In [ ]:
def write_yolo_label(label_path: Path, boxes_xywh: List[List[float]], W: int, H: int):
    label_path.parent.mkdir(parents=True, exist_ok=True)
    lines = []
    for x, y, w, h in boxes_xywh:
        if w < MIN_BOX_SIZE or h < MIN_BOX_SIZE:
            continue
        xc = (x + w / 2.0) / W
        yc = (y + h / 2.0) / H
        wn = w / W
        hn = h / H
        if 0 <= xc <= 1 and 0 <= yc <= 1 and wn > 0 and hn > 0:
            lines.append(f"0 {xc:.8f} {yc:.8f} {wn:.8f} {hn:.8f}")
    label_path.write_text("\n".join(lines) + ("\n" if lines else ""))

def build_yolo_dataset(train_df: pd.DataFrame, val_df: pd.DataFrame, root: Path) -> Path:
    root = Path(root)
    yaml_path = root / "soccernet_ball.yaml"
    index_path = root / "index.csv"
    if not REBUILD_YOLO_DATASET and yaml_path.exists() and index_path.exists():
        print("Reusing cached YOLO dataset:", root)
        print("Set REBUILD_YOLO_DATASET=True to rebuild labels/links.")
        return yaml_path

    all_rows = []
    for split, df in [("train", train_df), ("val", val_df)]:
        for r in tqdm(df.itertuples(index=False), total=len(df), desc=f"YOLO {split}"):
            src = Path(r.image_path)
            # Preserve sequence name to avoid frame-name collisions across sequences.
            img_dst = root / "images" / split / r.seq / src.name
            lab_dst = root / "labels" / split / r.seq / f"{src.stem}.txt"

            safe_link_or_copy(src, img_dst, symlink=SYMLINK_IMAGES)
            write_yolo_label(lab_dst, r.boxes, int(r.width), int(r.height))

            all_rows.append({
                "split": split,
                "seq": r.seq,
                "frame_id": int(r.frame_id),
                "image": str(img_dst),
                "label": str(lab_dst),
                "n_boxes": len(r.boxes),
            })

    index_df = pd.DataFrame(all_rows)
    index_df.to_csv(root / "index.csv", index=False)

    data_yaml = {
        "path": str(root),
        "train": "images/train",
        "val": "images/val",
        "names": {0: "ball"},
    }
    with open(yaml_path, "w") as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False)

    print("YOLO yaml:", yaml_path)
    print(index_df.groupby(["split"])["n_boxes"].agg(["count", "sum"]))
    return yaml_path

yolo_yaml_path = None
if "yolo" in RUN_DETECTORS:
    yolo_yaml_path = build_yolo_dataset(train_index, val_index, YOLO_DATA_ROOT)
else:
    print("Skipping YOLO dataset build because RUN_DETECTORS does not contain 'yolo'.")

In [ ]:
def optional_existing_path(p):
    if p is None:
        return None
    p = Path(p)
    return p if str(p) not in ["", "."] and p.exists() else None

def train_or_load_yolo():
    from ultralytics import YOLO

    existing = optional_existing_path(YOLO_EXISTING_CKPT)
    if existing is not None:
        print("Loading existing YOLO checkpoint:", existing)
        return YOLO(str(existing)), existing

    run_name = f"soccernet_ball_{Path(YOLO_WEIGHTS).stem}_{YOLO_IMGSZ}"
    run_dir = RUNS_ROOT / run_name
    best_pt = run_dir / "weights" / "best.pt"
    last_pt = run_dir / "weights" / "last.pt"

    if best_pt.exists() or last_pt.exists():
        ckpt = best_pt if best_pt.exists() else last_pt
        print("Reusing YOLO checkpoint:", ckpt)
        return YOLO(str(ckpt)), ckpt

    model = YOLO(YOLO_WEIGHTS)
    model.train(
        data=str(yolo_yaml_path),
        imgsz=YOLO_IMGSZ,
        epochs=YOLO_EPOCHS,
        batch=YOLO_BATCH,
        device=YOLO_DEVICE_TRAIN,
        workers=YOLO_WORKERS,
        patience=15,
        close_mosaic=10,
        project=str(RUNS_ROOT),
        name=run_name,
        exist_ok=True,
    )

    ckpt = best_pt if best_pt.exists() else last_pt
    print("YOLO checkpoint:", ckpt, "exists=", ckpt.exists())
    return YOLO(str(ckpt)), ckpt

yolo_model, yolo_ckpt = (None, None)
if "yolo" in RUN_DETECTORS:
    yolo_model, yolo_ckpt = train_or_load_yolo()

## 4. Train/load DETR detector

This section is skipped automatically when `RUN_DETECTORS` does not contain `"detr"`.


### DETR compatibility note

This cell forces the Hugging Face DETR image processor to use `use_fast=False` when possible. Some Kaggle/Transformers versions load `DetrImageProcessorFast` by default, and its `pad()` API can reject `return_tensors="pt"`. The collate function below also includes a manual padding fallback.


In [ ]:
class SoccerNetBallDetrDataset(Dataset):
    def __init__(self, frame_df: pd.DataFrame, processor):
        self.df = frame_df.reset_index(drop=True)
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        image = Image.open(r["image_path"]).convert("RGB")
        annotations = []
        for box in r["boxes"]:
            x, y, w, h = [float(v) for v in box]
            if w < MIN_BOX_SIZE or h < MIN_BOX_SIZE:
                continue
            annotations.append({
                "bbox": [x, y, w, h],
                "category_id": 0,
                "area": float(w * h),
                "iscrowd": 0,
            })

        target = {
            "image_id": int(idx),
            "annotations": annotations,
        }
        encoding = self.processor(images=image, annotations=target, return_tensors="pt")
        item = {
            "pixel_values": encoding["pixel_values"].squeeze(0),
            "labels": encoding["labels"][0],
        }
        return item

def load_detr_image_processor(path_or_name):
    """
    Kaggle/Transformers may load DetrImageProcessorFast by default.
    Some versions of DetrImageProcessorFast.pad() do not accept return_tensors="pt",
    which breaks the DETR DataLoader collate_fn. Force slow processor when possible.
    """
    from transformers import AutoImageProcessor
    try:
        return AutoImageProcessor.from_pretrained(str(path_or_name), use_fast=False)
    except TypeError:
        # Older Transformers versions may not expose use_fast for this class.
        return AutoImageProcessor.from_pretrained(str(path_or_name))

def manual_pad_pixel_values(pixel_values):
    """
    Fallback padding for DETR when processor.pad(...) API changes.
    Input: list of tensors [C,H,W].
    Output:
      pixel_values: [B,C,Hmax,Wmax]
      pixel_mask:   [B,Hmax,Wmax], 1 for real pixels and 0 for padding.
    """
    if len(pixel_values) == 0:
        raise ValueError("Empty batch in DETR collate_fn")

    max_h = max(int(x.shape[-2]) for x in pixel_values)
    max_w = max(int(x.shape[-1]) for x in pixel_values)
    c = int(pixel_values[0].shape[0])

    padded = pixel_values[0].new_zeros((len(pixel_values), c, max_h, max_w))
    pixel_mask = torch.zeros((len(pixel_values), max_h, max_w), dtype=torch.long)

    for i, pv in enumerate(pixel_values):
        h, w = int(pv.shape[-2]), int(pv.shape[-1])
        padded[i, :, :h, :w] = pv
        pixel_mask[i, :h, :w] = 1

    return {"pixel_values": padded, "pixel_mask": pixel_mask}

def make_detr_collate_fn(processor):
    def collate_fn(batch):
        pixel_values = [item["pixel_values"] for item in batch]

        # Prefer the official processor padding, but fall back to manual padding
        # because DetrImageProcessorFast.pad() changed API in some Kaggle images.
        try:
            encoding = processor.pad(pixel_values, return_tensors="pt")
            pixel_values_batch = encoding["pixel_values"]
            pixel_mask_batch = encoding["pixel_mask"]
        except TypeError:
            encoding = manual_pad_pixel_values(pixel_values)
            pixel_values_batch = encoding["pixel_values"]
            pixel_mask_batch = encoding["pixel_mask"]

        labels = [item["labels"] for item in batch]
        return {
            "pixel_values": pixel_values_batch,
            "pixel_mask": pixel_mask_batch,
            "labels": labels,
        }
    return collate_fn

def move_detr_labels_to_device(labels, device):
    out = []
    for lab in labels:
        out.append({k: v.to(device) if hasattr(v, "to") else v for k, v in lab.items()})
    return out

def train_or_load_detr():
    from transformers import DetrForObjectDetection

    existing = optional_existing_path(DETR_EXISTING_DIR)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if existing is not None:
        print("Loading existing DETR checkpoint:", existing)
        processor = load_detr_image_processor(existing)
        model = DetrForObjectDetection.from_pretrained(str(existing)).to(device)
        model.eval()
        return model, processor, existing, device

    ckpt_dir = DETR_RUN_DIR / "checkpoint-final"
    if ckpt_dir.exists():
        print("Reusing DETR checkpoint:", ckpt_dir)
        processor = load_detr_image_processor(ckpt_dir)
        model = DetrForObjectDetection.from_pretrained(str(ckpt_dir)).to(device)
        model.eval()
        return model, processor, ckpt_dir, device

    processor = load_detr_image_processor(DETR_BASE_MODEL)
    model = DetrForObjectDetection.from_pretrained(
        DETR_BASE_MODEL,
        num_labels=1,
        id2label={0: "ball"},
        label2id={"ball": 0},
        ignore_mismatched_sizes=True,
    ).to(device)

    train_ds = SoccerNetBallDetrDataset(train_index, processor)
    val_ds = SoccerNetBallDetrDataset(val_index, processor)

    train_loader = DataLoader(
        train_ds,
        batch_size=DETR_BATCH,
        shuffle=True,
        num_workers=DETR_NUM_WORKERS,
        collate_fn=make_detr_collate_fn(processor),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=DETR_BATCH,
        shuffle=False,
        num_workers=DETR_NUM_WORKERS,
        collate_fn=make_detr_collate_fn(processor),
    )

    # Smaller LR for backbone; larger LR for detection head/transformer.
    backbone_params = []
    other_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "backbone" in name:
            backbone_params.append(param)
        else:
            other_params.append(param)

    optimizer = torch.optim.AdamW(
        [
            {"params": other_params, "lr": DETR_LR},
            {"params": backbone_params, "lr": DETR_BACKBONE_LR},
        ],
        weight_decay=DETR_WEIGHT_DECAY,
    )

    best_val = float("inf")
    for epoch in range(1, DETR_EPOCHS + 1):
        model.train()
        train_losses = []
        for batch in tqdm(train_loader, desc=f"DETR train epoch {epoch}/{DETR_EPOCHS}"):
            pixel_values = batch["pixel_values"].to(device)
            pixel_mask = batch["pixel_mask"].to(device)
            labels = move_detr_labels_to_device(batch["labels"], device)

            outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask, labels=labels)
            loss = outputs.loss

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"DETR val epoch {epoch}/{DETR_EPOCHS}"):
                pixel_values = batch["pixel_values"].to(device)
                pixel_mask = batch["pixel_mask"].to(device)
                labels = move_detr_labels_to_device(batch["labels"], device)
                outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask, labels=labels)
                val_losses.append(float(outputs.loss.detach().cpu()))

        mean_train = float(np.mean(train_losses)) if train_losses else None
        mean_val = float(np.mean(val_losses)) if val_losses else None
        print({"epoch": epoch, "train_loss": mean_train, "val_loss": mean_val})

        if mean_val is not None and mean_val < best_val:
            best_val = mean_val
            best_dir = DETR_RUN_DIR / "checkpoint-best"
            best_dir.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(best_dir)
            processor.save_pretrained(best_dir)
            print("Saved best DETR:", best_dir)

    ckpt_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(ckpt_dir)
    processor.save_pretrained(ckpt_dir)
    print("Saved final DETR:", ckpt_dir)

    model.eval()
    return model, processor, ckpt_dir, device

detr_model, detr_processor, detr_ckpt, detr_device = (None, None, None, None)
if "detr" in RUN_DETECTORS:
    detr_model, detr_processor, detr_ckpt, detr_device = train_or_load_detr()


## 5. Detector inference to candidate CSV

Each detector emits the same candidate schema:

`seq, frame_id, rank, x, y, w, h, cx, cy, score, cls, detector`

In [ ]:
def infer_sequence_candidates_yolo(
    model,
    seq_dir: Path,
    out_csv: Path,
    imgsz=1280,
    conf=0.005,
    iou=0.60,
    top_k=15,
    batch=16,
    device=0,
    reuse=True,
) -> pd.DataFrame:
    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    if reuse and out_csv.exists():
        return pd.read_csv(out_csv)

    info = parse_seqinfo(seq_dir)
    image_paths = [frame_path(seq_dir, fid) for fid in range(1, info["seq_length"] + 1)]
    image_paths = [p for p in image_paths if p.exists()]
    all_rows = []

    for start in tqdm(range(0, len(image_paths), batch), desc=f"YOLO infer {seq_dir.name}"):
        batch_paths = image_paths[start:start + batch]
        results = model.predict(
            source=[str(p) for p in batch_paths],
            imgsz=imgsz,
            conf=conf,
            iou=iou,
            device=device,
            verbose=False,
        )

        for p, res in zip(batch_paths, results):
            fid = int(Path(p).stem)
            if res.boxes is None or len(res.boxes) == 0:
                continue

            xyxy = res.boxes.xyxy.detach().cpu().numpy()
            scores = res.boxes.conf.detach().cpu().numpy()
            clss = res.boxes.cls.detach().cpu().numpy() if res.boxes.cls is not None else np.zeros(len(scores))

            order = np.argsort(-scores)[:top_k]
            for rank, idx in enumerate(order):
                x1, y1, x2, y2 = xyxy[idx]
                x, y, w, h = clip_box_xywh(x1, y1, x2 - x1, y2 - y1, info["im_width"], info["im_height"])
                if w <= 0 or h <= 0:
                    continue
                all_rows.append({
                    "seq": seq_dir.name,
                    "frame_id": fid,
                    "rank": rank,
                    "x": x,
                    "y": y,
                    "w": w,
                    "h": h,
                    "cx": x + w / 2.0,
                    "cy": y + h / 2.0,
                    "score": float(scores[idx]),
                    "cls": int(clss[idx]),
                    "detector": "yolo",
                })

    cand_df = pd.DataFrame(all_rows)
    if len(cand_df):
        cand_df = cand_df.sort_values(["frame_id", "score"], ascending=[True, False]).reset_index(drop=True)
    cand_df.to_csv(out_csv, index=False)
    return cand_df

def infer_sequence_candidates_detr(
    model,
    processor,
    device,
    seq_dir: Path,
    out_csv: Path,
    conf=0.005,
    top_k=15,
    batch=8,
    reuse=True,
) -> pd.DataFrame:
    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    if reuse and out_csv.exists():
        return pd.read_csv(out_csv)

    info = parse_seqinfo(seq_dir)
    image_paths = [frame_path(seq_dir, fid) for fid in range(1, info["seq_length"] + 1)]
    image_paths = [p for p in image_paths if p.exists()]
    all_rows = []

    model.eval()
    for start in tqdm(range(0, len(image_paths), batch), desc=f"DETR infer {seq_dir.name}"):
        batch_paths = image_paths[start:start + batch]
        images = [Image.open(p).convert("RGB") for p in batch_paths]
        inputs = processor(images=images, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        target_sizes = torch.tensor([img.size[::-1] for img in images], device=device)  # (h, w)
        results = processor.post_process_object_detection(
            outputs,
            threshold=conf,
            target_sizes=target_sizes,
        )

        for p, res in zip(batch_paths, results):
            fid = int(Path(p).stem)
            boxes = res["boxes"].detach().cpu().numpy() if len(res["boxes"]) else np.zeros((0, 4))
            scores = res["scores"].detach().cpu().numpy() if len(res["scores"]) else np.array([])
            labels = res["labels"].detach().cpu().numpy() if len(res["labels"]) else np.array([])

            order = np.argsort(-scores)[:top_k]
            for rank, idx in enumerate(order):
                x1, y1, x2, y2 = boxes[idx]
                x, y, w, h = clip_box_xywh(x1, y1, x2 - x1, y2 - y1, info["im_width"], info["im_height"])
                if w <= 0 or h <= 0:
                    continue
                all_rows.append({
                    "seq": seq_dir.name,
                    "frame_id": fid,
                    "rank": rank,
                    "x": x,
                    "y": y,
                    "w": w,
                    "h": h,
                    "cx": x + w / 2.0,
                    "cy": y + h / 2.0,
                    "score": float(scores[idx]),
                    "cls": int(labels[idx]) if len(labels) else 0,
                    "detector": "detr",
                })

    cand_df = pd.DataFrame(all_rows)
    if len(cand_df):
        cand_df = cand_df.sort_values(["frame_id", "score"], ascending=[True, False]).reset_index(drop=True)
    cand_df.to_csv(out_csv, index=False)
    return cand_df

def infer_candidates(detector_name: str, seq_dir: Path, reuse=True) -> pd.DataFrame:
    out_csv = CAND_ROOT / detector_name / f"{seq_dir.name}_candidates.csv"
    if detector_name == "yolo":
        assert yolo_model is not None
        return infer_sequence_candidates_yolo(
            yolo_model,
            seq_dir,
            out_csv,
            imgsz=YOLO_IMGSZ,
            conf=INFER_CONF,
            iou=INFER_IOU,
            top_k=TOP_K,
            batch=INFER_BATCH,
            device=YOLO_DEVICE_INFER,
            reuse=reuse,
        )
    if detector_name == "detr":
        assert detr_model is not None and detr_processor is not None
        return infer_sequence_candidates_detr(
            detr_model,
            detr_processor,
            detr_device,
            seq_dir,
            out_csv,
            conf=INFER_CONF,
            top_k=TOP_K,
            batch=max(1, min(INFER_BATCH, 8)),
            reuse=reuse,
        )
    raise ValueError(f"Unknown detector: {detector_name}")

## 6. Tracker 1: Greedy online tracker

This is the tracking algorithm from the ball-detection notebook. It selects the best candidate frame-by-frame using confidence, predicted center, velocity consistency, and box-size stability.

In [ ]:
@dataclass
class GreedyTrackParams:
    max_init_score: float = 0.01
    max_center_jump: float = 180.0
    velocity_weight: float = 1.0
    score_weight: float = 80.0
    size_weight: float = 0.15
    gap_growth: float = 1.5
    max_gap_for_interp: int = 25
    smooth_window: int = 5

def choose_candidate_for_frame(
    frame_cands: pd.DataFrame,
    last_state: Optional[dict],
    params: GreedyTrackParams,
):
    if len(frame_cands) == 0:
        return None

    if last_state is None:
        return frame_cands.sort_values("score", ascending=False).iloc[0].to_dict()

    gap = int(frame_cands["frame_id"].iloc[0]) - int(last_state["frame_id"])
    gap = max(1, gap)

    pred_cx = last_state["cx"] + last_state.get("vx", 0.0) * gap
    pred_cy = last_state["cy"] + last_state.get("vy", 0.0) * gap

    allowed = params.max_center_jump * gap * params.gap_growth
    rows = []
    for _, c in frame_cands.iterrows():
        dc = math.hypot(float(c.cx) - pred_cx, float(c.cy) - pred_cy)

        prev_area = max(1.0, float(last_state["w"]) * float(last_state["h"]))
        area = max(1.0, float(c.w) * float(c.h))
        size_cost = abs(math.log(area / prev_area))

        cost = (
            params.velocity_weight * dc
            + params.size_weight * size_cost
            - params.score_weight * float(c.score)
        )
        rows.append((cost, dc, c.to_dict()))

    rows.sort(key=lambda t: t[0])
    best_cost, best_dist, best = rows[0]

    if best_dist > allowed and best["score"] < 0.25:
        return None

    return best

def greedy_track(cand_df: pd.DataFrame, seq_len: int, params=GreedyTrackParams()) -> pd.DataFrame:
    if cand_df is None or len(cand_df) == 0:
        return pd.DataFrame(columns=["frame_id", "track_id", "x", "y", "w", "h", "score", "detected"])

    cand_df = cand_df.copy()
    cand_df = cand_df.sort_values(["frame_id", "rank", "score"], ascending=[True, True, False])

    track_rows = []
    last_state = None

    for fid in range(1, seq_len + 1):
        fc = cand_df[cand_df["frame_id"] == fid]
        chosen = choose_candidate_for_frame(fc, last_state, params)

        if chosen is None:
            track_rows.append({
                "frame_id": fid,
                "track_id": 1,
                "x": np.nan, "y": np.nan, "w": np.nan, "h": np.nan,
                "score": 0.0,
                "detected": 0,
            })
            continue

        chosen = dict(chosen)
        if last_state is not None:
            gap = max(1, fid - int(last_state["frame_id"]))
            chosen["vx"] = (chosen["cx"] - last_state["cx"]) / gap
            chosen["vy"] = (chosen["cy"] - last_state["cy"]) / gap
        else:
            chosen["vx"] = 0.0
            chosen["vy"] = 0.0

        last_state = chosen
        track_rows.append({
            "frame_id": fid,
            "track_id": 1,
            "x": chosen["x"], "y": chosen["y"], "w": chosen["w"], "h": chosen["h"],
            "score": chosen["score"],
            "detected": 1,
        })

    tr = pd.DataFrame(track_rows)

    for col in ["x", "y", "w", "h"]:
        tr[col] = tr[col].interpolate(
            method="linear",
            limit=params.max_gap_for_interp,
            limit_direction="both",
        )

    win = params.smooth_window
    if win and win > 1:
        for col in ["x", "y", "w", "h"]:
            tr[col] = tr[col].rolling(win, center=True, min_periods=1).median()

    tr["score"] = tr["score"].fillna(0.0)
    tr["track_id"] = 1
    return tr

## 7. Tracker 2: Offline Viterbi / dynamic programming tracker

This is the tracking algorithm from the second notebook. It solves a global path problem across the whole sequence.

In [ ]:
@dataclass
class ViterbiTrackParams:
    top_k: int = 15

    # Detection/emission cost.
    score_weight: float = 5.0
    rank_weight: float = 0.15
    miss_cost: float = 18.0

    # Motion/transition cost.
    max_center_jump: float = 220.0
    distance_weight: float = 12.0
    size_weight: float = 2.0
    miss_transition_cost: float = 2.0
    impossible_cost: float = 1e6
    strong_score: float = 0.35

    # Post-processing.
    max_gap_for_interp: int = 25
    smooth_window: int = 5

def _candidate_state(row):
    x, y, w, h = float(row.x), float(row.y), float(row.w), float(row.h)
    return {
        "frame_id": int(row.frame_id),
        "rank": int(row.rank),
        "x": x,
        "y": y,
        "w": w,
        "h": h,
        "cx": x + w / 2.0,
        "cy": y + h / 2.0,
        "score": float(row.score),
        "detected": 1,
    }

def _miss_state(fid):
    return {
        "frame_id": int(fid),
        "rank": 999,
        "x": np.nan,
        "y": np.nan,
        "w": np.nan,
        "h": np.nan,
        "cx": np.nan,
        "cy": np.nan,
        "score": 0.0,
        "detected": 0,
    }

def _emission_cost(state, p: ViterbiTrackParams):
    if state["detected"] == 0:
        return p.miss_cost
    score = max(float(state["score"]), 1e-9)
    return -p.score_weight * math.log(score) + p.rank_weight * float(state["rank"])

def _transition_cost(prev, cur, p: ViterbiTrackParams):
    if prev["detected"] == 0 or cur["detected"] == 0:
        return p.miss_transition_cost

    dx = float(cur["cx"]) - float(prev["cx"])
    dy = float(cur["cy"]) - float(prev["cy"])
    dist = math.hypot(dx, dy)

    if dist > p.max_center_jump and float(cur["score"]) < p.strong_score:
        return p.impossible_cost

    dist_cost = p.distance_weight * (dist / p.max_center_jump) ** 2

    prev_area = max(1.0, float(prev["w"]) * float(prev["h"]))
    cur_area = max(1.0, float(cur["w"]) * float(cur["h"]))
    size_cost = p.size_weight * abs(math.log(cur_area / prev_area))

    return dist_cost + size_cost

def viterbi_track(cand_df: pd.DataFrame, seq_len: int, params=ViterbiTrackParams()) -> pd.DataFrame:
    if cand_df is None or len(cand_df) == 0:
        return pd.DataFrame(columns=["frame_id", "track_id", "x", "y", "w", "h", "score", "detected"])

    cand_df = cand_df.copy()
    cand_df = cand_df.sort_values(["frame_id", "score"], ascending=[True, False])

    states_by_frame = []
    for fid in range(1, seq_len + 1):
        fc = cand_df[cand_df["frame_id"] == fid].sort_values("score", ascending=False).head(params.top_k)
        states = [_candidate_state(r) for r in fc.itertuples(index=False)]
        states.append(_miss_state(fid))
        states_by_frame.append(states)

    dp = []
    back = []

    first_states = states_by_frame[0]
    first_costs = np.array([_emission_cost(s, params) for s in first_states], dtype=np.float64)
    dp.append(first_costs)
    back.append(np.full(len(first_states), -1, dtype=np.int32))

    for t in tqdm(range(1, seq_len), desc="Viterbi tracking"):
        prev_states = states_by_frame[t - 1]
        cur_states = states_by_frame[t]

        cur_costs = np.full(len(cur_states), np.inf, dtype=np.float64)
        cur_back = np.full(len(cur_states), -1, dtype=np.int32)

        for j, cur in enumerate(cur_states):
            emit = _emission_cost(cur, params)
            best_i = -1
            best_cost = np.inf

            for i, prev in enumerate(prev_states):
                cost = dp[t - 1][i] + _transition_cost(prev, cur, params) + emit
                if cost < best_cost:
                    best_cost = cost
                    best_i = i

            cur_costs[j] = best_cost
            cur_back[j] = best_i

        dp.append(cur_costs)
        back.append(cur_back)

    last_idx = int(np.argmin(dp[-1]))
    selected = []

    for t in reversed(range(seq_len)):
        state = states_by_frame[t][last_idx]
        selected.append(state)
        last_idx = int(back[t][last_idx])
        if last_idx < 0 and t > 0:
            break

    selected = list(reversed(selected))
    tr = pd.DataFrame(selected)
    tr = tr[["frame_id", "x", "y", "w", "h", "score", "detected"]].copy()

    for col in ["x", "y", "w", "h"]:
        tr[col] = tr[col].interpolate(
            method="linear",
            limit=params.max_gap_for_interp,
            limit_direction="both",
        )

    if params.smooth_window and params.smooth_window > 1:
        for col in ["x", "y", "w", "h"]:
            tr[col] = tr[col].rolling(params.smooth_window, center=True, min_periods=1).median()

    tr["score"] = tr["score"].fillna(0.0)
    tr["detected"] = tr["detected"].astype(int)
    tr["track_id"] = 1
    return tr[["frame_id", "track_id", "x", "y", "w", "h", "score", "detected"]]

## 8. Trackers 3–4: SORT and OC-SORT-style SORT

These two trackers add the requested SORT-family baselines:

- `sort`: classic SORT idea — Kalman prediction + Hungarian association using IoU.
- `ocsort`: observation-centric SORT variant — keeps the SORT/Kalman structure but adds center-distance and observation-direction consistency, which is usually better for a tiny fast-moving ball where IoU between consecutive boxes can become zero.

Both implementations are self-contained and do not require external SORT/DeepSORT repositories.

In [ ]:

@dataclass
class SortTrackParams:
    top_k: int = 15
    score_thresh: float = 0.01
    iou_threshold: float = 0.05
    max_age: int = 15
    min_hits: int = 2
    max_tracks_per_frame: Optional[int] = None  # None = output all confirmed active tracks

@dataclass
class OCSortTrackParams:
    top_k: int = 15
    score_thresh: float = 0.01
    iou_threshold: float = 0.02
    max_center_dist: float = 220.0
    center_distance_weight: float = 0.35
    direction_weight: float = 0.25
    max_age: int = 20
    min_hits: int = 1
    max_tracks_per_frame: Optional[int] = None

def _xywh_to_sort_z(box_xywh):
    x, y, w, h = [float(v) for v in box_xywh]
    cx = x + w / 2.0
    cy = y + h / 2.0
    s = max(1e-6, w * h)
    r = max(1e-6, w / max(h, 1e-6))
    return np.array([[cx], [cy], [s], [r]], dtype=np.float64)

def _sort_x_to_xywh(x):
    cx, cy, s, r = float(x[0]), float(x[1]), float(x[2]), float(x[3])
    s = max(s, 1e-6)
    r = max(r, 1e-6)
    w = math.sqrt(s * r)
    h = s / max(w, 1e-6)
    return np.array([cx - w / 2.0, cy - h / 2.0, w, h], dtype=np.float64)

def _bbox_center_xywh(box):
    return np.array([float(box[0]) + float(box[2]) / 2.0, float(box[1]) + float(box[3]) / 2.0], dtype=np.float64)

class _KalmanBoxTrack:
    _next_id = 1

    def __init__(self, bbox_xywh, score, frame_id):
        # State: [cx, cy, area, ratio, vx, vy, v_area]^T
        self.x = np.zeros((7, 1), dtype=np.float64)
        self.x[:4] = _xywh_to_sort_z(bbox_xywh)

        self.F = np.eye(7, dtype=np.float64)
        self.F[0, 4] = 1.0
        self.F[1, 5] = 1.0
        self.F[2, 6] = 1.0

        self.H = np.zeros((4, 7), dtype=np.float64)
        self.H[0, 0] = 1.0
        self.H[1, 1] = 1.0
        self.H[2, 2] = 1.0
        self.H[3, 3] = 1.0

        self.P = np.eye(7, dtype=np.float64)
        self.P[4:, 4:] *= 1000.0
        self.P *= 10.0
        self.R = np.eye(4, dtype=np.float64)
        self.R[2:, 2:] *= 10.0
        self.Q = np.eye(7, dtype=np.float64) * 0.01
        self.Q[4:, 4:] *= 0.01

        self.id = _KalmanBoxTrack._next_id
        _KalmanBoxTrack._next_id += 1

        self.age = 0
        self.hits = 1
        self.hit_streak = 1
        self.time_since_update = 0
        self.last_score = float(score)
        self.last_observation = np.asarray(bbox_xywh, dtype=np.float64)
        self.last_observation_frame = int(frame_id)
        self.obs_velocity = np.zeros(2, dtype=np.float64)

    def predict(self):
        if self.x[2] + self.x[6] <= 0:
            self.x[6] = 0.0
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        self.age += 1
        if self.time_since_update > 0:
            self.hit_streak = 0
        self.time_since_update += 1
        return self.get_state()

    def update(self, bbox_xywh, score, frame_id):
        bbox_xywh = np.asarray(bbox_xywh, dtype=np.float64)
        z = _xywh_to_sort_z(bbox_xywh)

        # Observation-centric velocity estimate for OC-SORT-style association.
        dt = max(1, int(frame_id) - int(self.last_observation_frame))
        prev_center = _bbox_center_xywh(self.last_observation)
        cur_center = _bbox_center_xywh(bbox_xywh)
        self.obs_velocity = (cur_center - prev_center) / float(dt)

        y = z - (self.H @ self.x)
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        I = np.eye(self.P.shape[0], dtype=np.float64)
        self.P = (I - K @ self.H) @ self.P

        self.time_since_update = 0
        self.hits += 1
        self.hit_streak += 1
        self.last_score = float(score)
        self.last_observation = bbox_xywh
        self.last_observation_frame = int(frame_id)

    def get_state(self):
        return _sort_x_to_xywh(self.x.reshape(-1))

    def obs_direction(self):
        norm = float(np.linalg.norm(self.obs_velocity))
        if norm < 1e-6:
            return None
        return self.obs_velocity / norm

def _sort_associate_iou(dets_xywh, trk_xywh, iou_thr):
    if len(trk_xywh) == 0:
        return [], list(range(len(dets_xywh))), []
    if len(dets_xywh) == 0:
        return [], [], list(range(len(trk_xywh)))

    ious = iou_matrix_xywh(dets_xywh, trk_xywh)
    rows, cols = linear_sum_assignment(-ious)

    matches = []
    unmatched_dets = set(range(len(dets_xywh)))
    unmatched_trks = set(range(len(trk_xywh)))
    for d, t in zip(rows, cols):
        if float(ious[d, t]) >= float(iou_thr):
            matches.append((int(d), int(t)))
            unmatched_dets.discard(int(d))
            unmatched_trks.discard(int(t))
    return matches, sorted(unmatched_dets), sorted(unmatched_trks)

def _ocsort_associate(dets_xywh, trks, trk_xywh, params: OCSortTrackParams):
    if len(trk_xywh) == 0:
        return [], list(range(len(dets_xywh))), []
    if len(dets_xywh) == 0:
        return [], [], list(range(len(trk_xywh)))

    dets_xywh = np.asarray(dets_xywh, dtype=np.float64)
    trk_xywh = np.asarray(trk_xywh, dtype=np.float64)
    ious = iou_matrix_xywh(dets_xywh, trk_xywh)

    det_centers = np.stack([_bbox_center_xywh(b) for b in dets_xywh], axis=0)
    trk_centers = np.stack([_bbox_center_xywh(b) for b in trk_xywh], axis=0)
    center_dist = np.linalg.norm(det_centers[:, None, :] - trk_centers[None, :, :], axis=2)
    center_cost = np.minimum(center_dist / max(params.max_center_dist, 1e-6), 5.0)

    direction_cost = np.zeros_like(center_cost)
    for j, trk in enumerate(trks):
        obs_dir = trk.obs_direction()
        if obs_dir is None:
            continue
        vec = det_centers - _bbox_center_xywh(trk.last_observation)[None, :]
        norm = np.linalg.norm(vec, axis=1)
        valid = norm > 1e-6
        cand_dir = np.zeros_like(vec)
        cand_dir[valid] = vec[valid] / norm[valid, None]
        cos = np.clip(cand_dir @ obs_dir, -1.0, 1.0)
        direction_cost[:, j] = (1.0 - cos) / 2.0
        direction_cost[~valid, j] = 0.5

    cost = (1.0 - ious) + params.center_distance_weight * center_cost + params.direction_weight * direction_cost
    rows, cols = linear_sum_assignment(cost)

    matches = []
    unmatched_dets = set(range(len(dets_xywh)))
    unmatched_trks = set(range(len(trk_xywh)))
    for d, t in zip(rows, cols):
        valid_iou = float(ious[d, t]) >= float(params.iou_threshold)
        valid_dist = float(center_dist[d, t]) <= float(params.max_center_dist)
        if valid_iou or valid_dist:
            matches.append((int(d), int(t)))
            unmatched_dets.discard(int(d))
            unmatched_trks.discard(int(t))
    return matches, sorted(unmatched_dets), sorted(unmatched_trks)

def _run_sort_family(cand_df: pd.DataFrame, seq_len: int, params, mode: str) -> pd.DataFrame:
    if cand_df is None or len(cand_df) == 0:
        return pd.DataFrame(columns=["frame_id", "track_id", "x", "y", "w", "h", "score", "detected"])

    _KalmanBoxTrack._next_id = 1
    cand_df = cand_df.copy()
    cand_df = cand_df.sort_values(["frame_id", "score"], ascending=[True, False])
    tracks = []
    out_rows = []

    for fid in tqdm(range(1, seq_len + 1), desc=f"{mode.upper()} tracking"):
        fc = cand_df[cand_df["frame_id"] == fid].sort_values("score", ascending=False).head(params.top_k)
        fc = fc[fc["score"] >= params.score_thresh]
        dets = fc[["x", "y", "w", "h", "score"]].values.astype(np.float64) if len(fc) else np.zeros((0, 5), dtype=np.float64)

        predicted_boxes = []
        alive_tracks = []
        for trk in tracks:
            pred = trk.predict()
            if np.isfinite(pred).all() and pred[2] > 0 and pred[3] > 0:
                predicted_boxes.append(pred)
                alive_tracks.append(trk)
        tracks = alive_tracks
        predicted_boxes = np.asarray(predicted_boxes, dtype=np.float64).reshape(-1, 4)

        if mode == "sort":
            matches, unmatched_dets, unmatched_trks = _sort_associate_iou(
                dets[:, :4] if len(dets) else np.zeros((0, 4)),
                predicted_boxes,
                params.iou_threshold,
            )
        elif mode == "ocsort":
            matches, unmatched_dets, unmatched_trks = _ocsort_associate(
                dets[:, :4] if len(dets) else np.zeros((0, 4)),
                tracks,
                predicted_boxes,
                params,
            )
        else:
            raise ValueError(f"Unknown SORT-family mode: {mode}")

        for det_idx, trk_idx in matches:
            bbox = dets[det_idx, :4]
            score = float(dets[det_idx, 4])
            tracks[trk_idx].update(bbox, score, frame_id=fid)

        for det_idx in unmatched_dets:
            bbox = dets[det_idx, :4]
            score = float(dets[det_idx, 4])
            tracks.append(_KalmanBoxTrack(bbox, score, frame_id=fid))

        # Remove old tracks after updates/new-track creation.
        tracks = [t for t in tracks if t.time_since_update <= params.max_age]

        frame_outputs = []
        for trk in tracks:
            confirmed = (trk.hits >= params.min_hits) or (fid <= params.min_hits)
            if not confirmed:
                continue
            # Standard SORT-style output: only tracks updated in the current frame.
            if trk.time_since_update != 0:
                continue
            x, y, w, h = trk.get_state()
            if not np.isfinite([x, y, w, h]).all() or w <= 0 or h <= 0:
                continue
            frame_outputs.append({
                "frame_id": int(fid),
                "track_id": int(trk.id),
                "x": float(x),
                "y": float(y),
                "w": float(w),
                "h": float(h),
                "score": float(trk.last_score),
                "detected": 1,
            })

        frame_outputs = sorted(frame_outputs, key=lambda r: r["score"], reverse=True)
        if params.max_tracks_per_frame is not None:
            frame_outputs = frame_outputs[: int(params.max_tracks_per_frame)]
        out_rows.extend(frame_outputs)

    return pd.DataFrame(out_rows, columns=["frame_id", "track_id", "x", "y", "w", "h", "score", "detected"])

def sort_track(cand_df: pd.DataFrame, seq_len: int, params=SortTrackParams()) -> pd.DataFrame:
    return _run_sort_family(cand_df, seq_len=seq_len, params=params, mode="sort")

def ocsort_track(cand_df: pd.DataFrame, seq_len: int, params=OCSortTrackParams()) -> pd.DataFrame:
    return _run_sort_family(cand_df, seq_len=seq_len, params=params, mode="ocsort")


## 9. MOT export and evaluation metrics

The notebook computes:

- Standard MOT-style counts: `TP`, `FP`, `FN`, `IDSW`
- `MOTA`
- `MOTP_IoU`
- `IDF1`
- `Precision`, `Recall`
- Center error / hit rate
- HOTA-family metrics: `HOTA`, `DetA`, `AssA`, `LocA`

The HOTA implementation here is self-contained so the notebook can run without external benchmark tooling. For official challenge reporting, also keep the exported MOT `.txt` files and run the official evaluator if provided.

In [ ]:
def export_mot_ball(track_df: pd.DataFrame, out_txt: Path, track_id=None, export_interpolated=True):
    out_txt = Path(out_txt)
    out_txt.parent.mkdir(parents=True, exist_ok=True)

    lines = []
    for _, r in track_df.iterrows():
        if not np.isfinite([r.x, r.y, r.w, r.h]).all():
            continue
        if not export_interpolated and int(r.detected) == 0:
            continue

        row_tid = int(r.track_id) if track_id is None and "track_id" in track_df.columns else int(track_id or 1)
        lines.append(
            f"{int(r.frame_id)},{row_tid},"
            f"{float(r.x):.2f},{float(r.y):.2f},{float(r.w):.2f},{float(r.h):.2f},"
            f"{float(r.score):.6f},-1,-1,-1"
        )

    out_txt.write_text("\n".join(lines) + ("\n" if lines else ""))
    return out_txt

def prepare_pred_for_eval(track_df: pd.DataFrame) -> pd.DataFrame:
    pred = track_df.copy()
    pred = pred[np.isfinite(pred[["x", "y", "w", "h"]]).all(axis=1)].copy()
    if "track_id" not in pred.columns:
        pred["track_id"] = 1
    pred["track_id"] = pred["track_id"].astype(int)
    pred["frame_id"] = pred["frame_id"].astype(int)
    return pred[["frame_id", "track_id", "x", "y", "w", "h", "score", "detected"]]

def match_frame(gt_f: pd.DataFrame, pred_f: pd.DataFrame, iou_thr: float):
    gt_boxes = gt_f[["x", "y", "w", "h"]].values.astype(float)
    pred_boxes = pred_f[["x", "y", "w", "h"]].values.astype(float)
    ious = iou_matrix_xywh(gt_boxes, pred_boxes)

    if len(gt_f) == 0 or len(pred_f) == 0:
        return [], list(range(len(gt_f))), list(range(len(pred_f)))

    cost = -ious
    gi, pi = linear_sum_assignment(cost)

    matches = []
    matched_g = set()
    matched_p = set()
    for g_idx, p_idx in zip(gi, pi):
        iou = float(ious[g_idx, p_idx])
        if iou >= iou_thr:
            matches.append((g_idx, p_idx, iou))
            matched_g.add(g_idx)
            matched_p.add(p_idx)

    unmatched_g = [i for i in range(len(gt_f)) if i not in matched_g]
    unmatched_p = [i for i in range(len(pred_f)) if i not in matched_p]
    return matches, unmatched_g, unmatched_p

def evaluate_mot_basic(track_df: pd.DataFrame, seq_dir: Path, iou_thr=0.50) -> Dict[str, Any]:
    gt = load_ball_gt_for_seq(seq_dir)
    pred = prepare_pred_for_eval(track_df)
    info = parse_seqinfo(seq_dir)

    if len(gt) == 0:
        return {"eval_error": "No ball GT found"}

    TP = FP = FN = IDSW = 0
    iou_sum = 0.0
    match_count = 0

    last_match_for_gt = {}
    pair_match_counts = {}
    gt_id_counts = gt.groupby("track_id").size().to_dict()
    pred_id_counts = pred.groupby("track_id").size().to_dict()

    for fid in range(1, info["seq_length"] + 1):
        gt_f = gt[gt["frame_id"] == fid].reset_index(drop=True)
        pred_f = pred[pred["frame_id"] == fid].reset_index(drop=True)

        matches, ug, up = match_frame(gt_f, pred_f, iou_thr=iou_thr)
        TP += len(matches)
        FN += len(ug)
        FP += len(up)

        for g_idx, p_idx, iou in matches:
            gt_id = int(gt_f.iloc[g_idx]["track_id"])
            pred_id = int(pred_f.iloc[p_idx]["track_id"])

            if gt_id in last_match_for_gt and last_match_for_gt[gt_id] != pred_id:
                IDSW += 1
            last_match_for_gt[gt_id] = pred_id

            pair_match_counts[(gt_id, pred_id)] = pair_match_counts.get((gt_id, pred_id), 0) + 1
            iou_sum += iou
            match_count += 1

    total_gt = int(len(gt))
    total_pred = int(len(pred))

    # IDF1 via global identity assignment over pair-level matched detections.
    gt_ids = sorted(gt_id_counts)
    pred_ids = sorted(pred_id_counts)
    if gt_ids and pred_ids:
        mat = np.zeros((len(gt_ids), len(pred_ids)), dtype=np.float64)
        for (gid, pid), cnt in pair_match_counts.items():
            if gid in gt_ids and pid in pred_ids:
                mat[gt_ids.index(gid), pred_ids.index(pid)] = cnt
        rows, cols = linear_sum_assignment(-mat)
        IDTP = int(mat[rows, cols].sum())
    else:
        IDTP = 0
    IDFP = total_pred - IDTP
    IDFN = total_gt - IDTP
    IDF1 = (2 * IDTP / max(1, 2 * IDTP + IDFP + IDFN))

    precision = TP / max(1, TP + FP)
    recall = TP / max(1, TP + FN)
    mota = 1.0 - (FN + FP + IDSW) / max(1, total_gt)
    motp_iou = iou_sum / max(1, match_count)

    return {
        "GT": total_gt,
        "Pred": total_pred,
        "TP": int(TP),
        "FP": int(FP),
        "FN": int(FN),
        "IDSW": int(IDSW),
        "Precision": float(precision),
        "Recall": float(recall),
        "MOTA": float(mota),
        "MOTP_IoU": float(motp_iou),
        "IDTP": int(IDTP),
        "IDFP": int(IDFP),
        "IDFN": int(IDFN),
        "IDF1": float(IDF1),
    }

def evaluate_center_error(track_df: pd.DataFrame, seq_dir: Path, thr_px=20) -> Dict[str, Any]:
    gt = load_ball_gt_for_seq(seq_dir)
    pred = prepare_pred_for_eval(track_df)

    if len(gt) == 0:
        return {"center_eval_error": "No ball GT found"}

    # For single-ball sequences, use largest GT ball box per frame if duplicates exist.
    gt2 = gt.copy()
    gt2["area"] = gt2["w"] * gt2["h"]
    gt2 = gt2.sort_values("area", ascending=False).drop_duplicates("frame_id")
    gt2 = gt2.drop(columns=["area"])

    pred2 = pred.copy()
    pred2["area"] = pred2["w"] * pred2["h"]
    pred2 = pred2.sort_values(["frame_id", "score", "area"], ascending=[True, False, False]).drop_duplicates("frame_id")
    pred2["cx"] = pred2["x"] + pred2["w"] / 2
    pred2["cy"] = pred2["y"] + pred2["h"] / 2

    m = gt2.merge(
        pred2[["frame_id", "x", "y", "w", "h", "cx", "cy", "detected"]],
        on="frame_id",
        how="left",
        suffixes=("_gt", "_pred"),
    )

    pred_ok = np.isfinite(m["cx_pred"]) & np.isfinite(m["cy_pred"])
    err = np.sqrt((m["cx_pred"] - m["cx_gt"]) ** 2 + (m["cy_pred"] - m["cy_gt"]) ** 2)
    valid_err = err[pred_ok]

    return {
        "gt_frames": int(len(gt2)),
        "predicted_on_gt_frames": int(pred_ok.sum()),
        "coverage": float(pred_ok.mean()) if len(m) else 0.0,
        f"hit@{thr_px}px": float((valid_err <= thr_px).mean()) if len(valid_err) else 0.0,
        "mean_center_error": float(valid_err.mean()) if len(valid_err) else np.nan,
        "median_center_error": float(valid_err.median()) if len(valid_err) else np.nan,
        "p90_center_error": float(valid_err.quantile(0.90)) if len(valid_err) else np.nan,
    }

def evaluate_hota(track_df: pd.DataFrame, seq_dir: Path, thresholds=HOTA_THRESHOLDS) -> Dict[str, Any]:
    gt = load_ball_gt_for_seq(seq_dir)
    pred = prepare_pred_for_eval(track_df)
    info = parse_seqinfo(seq_dir)

    if len(gt) == 0:
        return {"hota_eval_error": "No ball GT found"}

    gt_total_by_id = gt.groupby("track_id").size().to_dict()
    pred_total_by_id = pred.groupby("track_id").size().to_dict()

    alpha_rows = []
    for alpha in thresholds:
        TP = FP = FN = 0
        loc_sum = 0.0
        matched_pairs = []

        pair_tp = {}

        # First pass: frame-level matches and pair counts.
        for fid in range(1, info["seq_length"] + 1):
            gt_f = gt[gt["frame_id"] == fid].reset_index(drop=True)
            pred_f = pred[pred["frame_id"] == fid].reset_index(drop=True)

            matches, ug, up = match_frame(gt_f, pred_f, iou_thr=float(alpha))
            TP += len(matches)
            FN += len(ug)
            FP += len(up)

            for g_idx, p_idx, iou in matches:
                gt_id = int(gt_f.iloc[g_idx]["track_id"])
                pred_id = int(pred_f.iloc[p_idx]["track_id"])
                pair = (gt_id, pred_id)
                pair_tp[pair] = pair_tp.get(pair, 0) + 1
                matched_pairs.append(pair)
                loc_sum += float(iou)

        det_a = TP / max(1, TP + FP + FN)
        loc_a = loc_sum / max(1, TP)

        # Association accuracy averaged over matched detections.
        ass_values = []
        for pair in matched_pairs:
            gid, pid = pair
            tpa = pair_tp[pair]
            fna = gt_total_by_id.get(gid, 0) - tpa
            fpa = pred_total_by_id.get(pid, 0) - tpa
            ass_a_pair = tpa / max(1, tpa + fna + fpa)
            ass_values.append(ass_a_pair)

        ass_a = float(np.mean(ass_values)) if ass_values else 0.0
        hota = math.sqrt(det_a * ass_a) if det_a > 0 and ass_a > 0 else 0.0

        alpha_rows.append({
            "alpha": float(alpha),
            "HOTA_alpha": float(hota),
            "DetA_alpha": float(det_a),
            "AssA_alpha": float(ass_a),
            "LocA_alpha": float(loc_a),
            "TP_alpha": int(TP),
            "FP_alpha": int(FP),
            "FN_alpha": int(FN),
        })

    df = pd.DataFrame(alpha_rows)
    return {
        "HOTA": float(df["HOTA_alpha"].mean()),
        "DetA": float(df["DetA_alpha"].mean()),
        "AssA": float(df["AssA_alpha"].mean()),
        "LocA": float(df["LocA_alpha"].mean()),
        "HOTA@0.50": float(df.loc[np.isclose(df["alpha"], 0.50), "HOTA_alpha"].iloc[0]) if any(np.isclose(df["alpha"], 0.50)) else np.nan,
    }

def evaluate_all_metrics(track_df: pd.DataFrame, seq_dir: Path) -> Dict[str, Any]:
    out = {}
    out.update(evaluate_mot_basic(track_df, seq_dir, iou_thr=MOT_IOU_THRESHOLD))
    out.update(evaluate_center_error(track_df, seq_dir, thr_px=CENTER_HIT_PX))
    out.update(evaluate_hota(track_df, seq_dir, thresholds=HOTA_THRESHOLDS))
    return out

## 10. Run YOLO 1?4 experiment + inference-time benchmark

This cell is the main one-run benchmark. It measures both quality and speed for `yolo` with four trackers.

Timing columns:

- `detection_time_sec`: candidate generation time for one detector on one sequence.
- `tracker_time_sec`: tracker runtime on the detector candidates for one sequence.
- `total_inference_time_sec`: `detection_time_sec + tracker_time_sec`.
- `detect_fps`, `track_fps`, `end_to_end_fps`: throughput based on `seq_num_frames`.
- `candidate_source`: `model_inference` or `cache_csv`. Use `model_inference` for real detector benchmark.

Outputs are saved under `WORK_ROOT`:

```python
WORK_ROOT
RESULTS_ROOT
```

On local Windows this resolves to `outputs/ball_tracking/yolo_1x4_tracking_benchmark`. On Kaggle it resolves under the Kaggle working directory so the artifact can be downloaded.


In [ ]:
def _sync_for_timing():
    if TIMING_SYNC_CUDA and torch.cuda.is_available():
        torch.cuda.synchronize()

def timed_call(fn, *args, **kwargs):
    _sync_for_timing()
    t0 = time.perf_counter()
    result = fn(*args, **kwargs)
    _sync_for_timing()
    return result, time.perf_counter() - t0

def safe_fps(n_frames: int, seconds: float):
    seconds = float(seconds)
    if seconds <= 0 or not np.isfinite(seconds):
        return np.nan
    return float(n_frames) / seconds

def run_tracker(tracker_name: str, cand_df: pd.DataFrame, seq_len: int) -> pd.DataFrame:
    if tracker_name == "greedy":
        return greedy_track(cand_df, seq_len=seq_len, params=GreedyTrackParams())
    if tracker_name == "viterbi":
        return viterbi_track(cand_df, seq_len=seq_len, params=ViterbiTrackParams(top_k=TOP_K))
    if tracker_name == "sort":
        return sort_track(cand_df, seq_len=seq_len, params=SortTrackParams(top_k=TOP_K))
    if tracker_name == "ocsort":
        return ocsort_track(cand_df, seq_len=seq_len, params=OCSortTrackParams(top_k=TOP_K))
    raise ValueError(f"Unknown tracker: {tracker_name}")

def run_full_experiment(
    detectors=RUN_DETECTORS,
    trackers=RUN_TRACKERS,
    seqs_to_eval=eval_seqs,
    reuse_candidates=REUSE_CANDIDATES_FOR_BENCHMARK,
):
    rows = []

    for detector_name in detectors:
        for seq_dir in seqs_to_eval:
            info = parse_seqinfo(seq_dir)
            n_frames = int(info["seq_length"])
            candidate_csv = CAND_ROOT / detector_name / f"{seq_dir.name}_candidates.csv"
            candidate_source = "cache_csv" if reuse_candidates and candidate_csv.exists() else "model_inference"

            cand_df, detection_time_sec = timed_call(
                infer_candidates,
                detector_name,
                seq_dir,
                reuse=reuse_candidates,
            )
            detect_fps = safe_fps(n_frames, detection_time_sec)
            print(
                f"\n=== detector={detector_name} seq={seq_dir.name} "
                f"candidates={len(cand_df)} source={candidate_source} "
                f"detect_time={detection_time_sec:.3f}s detect_fps={detect_fps:.2f} ==="
            )

            for tracker_name in trackers:
                print(f"--- tracker={tracker_name} ---")
                track_df, tracker_time_sec = timed_call(
                    run_tracker,
                    tracker_name,
                    cand_df,
                    seq_len=n_frames,
                )
                total_inference_time_sec = detection_time_sec + tracker_time_sec

                out_dir = TRACK_ROOT / detector_name / tracker_name
                mot_dir = MOT_ROOT / detector_name / tracker_name
                out_dir.mkdir(parents=True, exist_ok=True)
                mot_dir.mkdir(parents=True, exist_ok=True)

                track_csv = out_dir / f"{seq_dir.name}_track.csv"
                mot_txt = mot_dir / f"{seq_dir.name}.txt"
                track_df.to_csv(track_csv, index=False)
                export_mot_ball(track_df, mot_txt, track_id=None, export_interpolated=True)

                metrics = evaluate_all_metrics(track_df, seq_dir)
                row = {
                    "detector": detector_name,
                    "tracker": tracker_name,
                    "seq": seq_dir.name,
                    "seq_num_frames": n_frames,
                    "candidate_rows": int(len(cand_df)),
                    "candidate_source": candidate_source,
                    "detection_time_sec": float(detection_time_sec),
                    "tracker_time_sec": float(tracker_time_sec),
                    "total_inference_time_sec": float(total_inference_time_sec),
                    "detect_fps": safe_fps(n_frames, detection_time_sec),
                    "track_fps": safe_fps(n_frames, tracker_time_sec),
                    "end_to_end_fps": safe_fps(n_frames, total_inference_time_sec),
                    "track_csv": str(track_csv),
                    "mot_txt": str(mot_txt),
                }
                row.update(metrics)
                rows.append(row)
                print({
                    k: row[k]
                    for k in [
                        "detector", "tracker", "seq", "HOTA", "MOTA", "IDF1",
                        "detection_time_sec", "tracker_time_sec", "total_inference_time_sec", "end_to_end_fps"
                    ]
                    if k in row
                })

    results = pd.DataFrame(rows)
    results_path = RESULTS_ROOT / f"per_sequence_{EXPERIMENT_TAG}_metrics.csv"
    results.to_csv(results_path, index=False)

    numeric_cols = [
        "GT", "Pred", "TP", "FP", "FN", "IDSW",
        "Precision", "Recall", "MOTA", "MOTP_IoU", "IDF1",
        "coverage", f"hit@{CENTER_HIT_PX}px",
        "mean_center_error", "median_center_error", "p90_center_error",
        "HOTA", "DetA", "AssA", "LocA", "HOTA@0.50",
        "seq_num_frames", "candidate_rows",
        "detection_time_sec", "tracker_time_sec", "total_inference_time_sec",
        "detect_fps", "track_fps", "end_to_end_fps",
    ]
    agg_dict = {}
    for c in numeric_cols:
        if c in results.columns:
            agg_dict[c] = "mean"

    summary = (
        results
        .groupby(["detector", "tracker"], as_index=False)
        .agg(agg_dict)
        .sort_values(["HOTA", "IDF1", "MOTA"], ascending=False)
    )
    summary_path = RESULTS_ROOT / f"summary_{EXPERIMENT_TAG}_metrics.csv"
    summary.to_csv(summary_path, index=False)

    timing_cols = [
        "detector", "tracker", "seq_num_frames",
        "detection_time_sec", "tracker_time_sec", "total_inference_time_sec",
        "detect_fps", "track_fps", "end_to_end_fps",
        "HOTA", "IDF1", "MOTA",
    ]
    timing_summary = summary[[c for c in timing_cols if c in summary.columns]].copy()
    timing_summary_path = RESULTS_ROOT / f"summary_{EXPERIMENT_TAG}_timing.csv"
    timing_summary.to_csv(timing_summary_path, index=False)

    print("Saved per-sequence metrics:", results_path)
    print("Saved summary:", summary_path)
    print("Saved timing summary:", timing_summary_path)
    return results, summary, timing_summary

results_df, summary_df, timing_summary_df = run_full_experiment()
display(summary_df)
display(timing_summary_df)
display(results_df.head())


## 11. Zip all outputs

In [ ]:
zip_base = WORK_ROOT.parent / f"{WORK_ROOT.name}_outputs"
zip_path = shutil.make_archive(str(zip_base), "zip", WORK_ROOT)
print("Created:", zip_path)


## 12. Optional: render one qualitative video

Change `SAMPLE_DETECTOR`, `SAMPLE_TRACKER`, and `SAMPLE_SEQ_NAME` to inspect a run.

In [ ]:
def render_track_video_h264(
    seq_dir: Path,
    track_df: pd.DataFrame,
    out_mp4: Path,
    max_frames=250,
    fps=None,
    width=960,
):
    out_mp4 = Path(out_mp4)
    out_mp4.parent.mkdir(parents=True, exist_ok=True)

    info = parse_seqinfo(seq_dir)
    seq_len = int(info["seq_length"])
    fps = fps or int(info["frame_rate"])

    tmp_dir = out_mp4.parent / f"{out_mp4.stem}_frames"
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)

    tr = track_df.set_index("frame_id")
    end_fid = seq_len if max_frames is None else min(seq_len, int(max_frames))

    for fid in tqdm(range(1, end_fid + 1), desc=f"render {seq_dir.name}"):
        img_path = frame_path(seq_dir, fid)
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        scale = width / img.shape[1]
        height = int(round(img.shape[0] * scale))
        img = cv2.resize(img, (width, height))

        if fid in tr.index:
            rr = tr.loc[fid]
            if isinstance(rr, pd.Series):
                rr = pd.DataFrame([rr])
            for _, r in rr.iterrows():
                if np.isfinite([r.x, r.y, r.w, r.h]).all():
                    x = int(round(float(r.x) * scale))
                    y = int(round(float(r.y) * scale))
                    w = int(round(float(r.w) * scale))
                    h = int(round(float(r.h) * scale))
                    cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 255), 2)
                    tid = int(r.track_id) if "track_id" in rr.columns else 1
                    txt = f"id={tid} {float(r.score):.2f}"
                    cv2.putText(img, txt, (x, max(20, y - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

        cv2.putText(img, f"{seq_dir.name} frame {fid}", (12, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.imwrite(str(tmp_dir / f"{fid:06d}.jpg"), img)

    cmd = [
        "ffmpeg", "-y",
        "-framerate", str(fps),
        "-i", str(tmp_dir / "%06d.jpg"),
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        str(out_mp4),
    ]
    subprocess.run(cmd, check=False, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    shutil.rmtree(tmp_dir, ignore_errors=True)
    return out_mp4

SAMPLE_DETECTOR = summary_df.iloc[0]["detector"] if len(summary_df) else RUN_DETECTORS[0]
SAMPLE_TRACKER = summary_df.iloc[0]["tracker"] if len(summary_df) else RUN_TRACKERS[0]
SAMPLE_SEQ_NAME = results_df.iloc[0]["seq"] if len(results_df) else eval_seqs[0].name

sample_seq = next(s for s in eval_seqs if s.name == SAMPLE_SEQ_NAME)
sample_track_csv = TRACK_ROOT / SAMPLE_DETECTOR / SAMPLE_TRACKER / f"{SAMPLE_SEQ_NAME}_track.csv"
sample_track_df = pd.read_csv(sample_track_csv)

sample_video = VIS_ROOT / f"{SAMPLE_DETECTOR}_{SAMPLE_TRACKER}_{SAMPLE_SEQ_NAME}.mp4"
render_track_video_h264(sample_seq, sample_track_df, sample_video, max_frames=250, width=960)
print("Video:", sample_video)
display(Video(str(sample_video), embed=True))

## 13. How to read the result

Primary ranking columns:

1. `HOTA`: balances detection quality and association quality.
2. `IDF1`: identity consistency, especially useful for SORT/OC-SORT where track IDs can fragment.
3. `MOTA`: old but common MOT summary metric.
4. `mean_center_error` / `hit@20px`: useful for tiny ball tracking.
5. `total_inference_time_sec` and `end_to_end_fps`: detector + tracker runtime.
6. `tracker_time_sec` / `track_fps`: tracker-only speed.
7. `detection_time_sec` / `detect_fps`: detector-only candidate generation speed.

Recommended quality comparison inside this notebook:

```python
summary_df[["detector", "tracker", "HOTA", "IDF1", "MOTA", "Recall", "Precision", "mean_center_error"]]
```

Recommended speed comparison inside this notebook:

```python
timing_summary_df[["detector", "tracker", "detection_time_sec", "tracker_time_sec", "total_inference_time_sec", "detect_fps", "track_fps", "end_to_end_fps"]]
```

Recommended combined comparison inside this notebook:

```python
summary_df[["detector", "tracker", "HOTA", "IDF1", "MOTA", "total_inference_time_sec", "end_to_end_fps"]]
```

To compare YOLO vs DETR after running both split notebooks, read the summaries from each notebook's output folder. Example:

```python
yolo_summary_path = RESULTS_ROOT / f"summary_{EXPERIMENT_TAG}_metrics.csv"
detr_summary_path = WORK_ROOT.parent / "detr_1x4_tracking_benchmark" / "results" / "summary_detr_1x4_metrics.csv"

frames = [pd.read_csv(yolo_summary_path)]
if detr_summary_path.exists():
    frames.append(pd.read_csv(detr_summary_path))
else:
    print("DETR summary not found:", detr_summary_path)

both = pd.concat(frames, ignore_index=True).sort_values(["HOTA", "IDF1", "MOTA"], ascending=False)
display(both)
```

Timing caveat: check `results_df["candidate_source"].unique()`. For real detector inference timing it should contain `"model_inference"`. If it contains `"cache_csv"`, set `REUSE_CANDIDATES_FOR_BENCHMARK = False` in Section 1 and rerun from candidate inference onward.
